## Aannames
#### 1. LT-data en event confidence uitsluiten

Long-term data: akkoord, maar met nuancering

Het uitsluiten van LT-data als target is methodologisch de correcte beslissing, en ze is goed te verantwoorden. Hieronder de academische redenering.

Academische formulering voor de thesis:

"De short-term (ST) en long-term (LT) bezettingsdata worden in dit onderzoek gescheiden gemodelleerd. De LT-data, afkomstig van abonnementhouders met habitueel en nagenoeg constant rijgedrag, vertoont structureel andere variantiepatronen dan het transiente ST-gebruik dat centraal staat in de onderzoeksvragen (Lira et al., 2021). De target-variabele van het predictiemodel is bijgevolg uitsluitend de ST-bezettingsgraad (occupancy_rate uit MAD_shortterm.parquet). LT-bezetting wordt niet als target opgenomen, maar kan in nb08 worden geëvalueerd als contextuele feature voor parkings waar het empirisch is vastgesteld dat LT-druk de beschikbare ST-ruimte structureel beperkt (P Hoogstraat, P Komet — zie nb07b, Sectie 2.3)."

Kanttekening die je niet moet vergeten: voor P Hoogstraat en P Komet geldt een ceiling effect: LT neemt respectievelijk ~72% en ~72% van de capaciteit in. De ST-bezettingsgraad is er fundamenteel beperkt in variabiliteit (maximum bereikbaar is ~28%). Dit verlaagt de theoretische voorspelbaarheid voor die parkings. Vermeld dit expliciet als een per-parking data-limitatie — niet als een reden om die parkings uit te sluiten, maar als context bij het interpreteren van hogere MAPE-waarden die je er waarschijnlijk zal zien in nb11.

#### 2. Event confidence: gedeeltelijk akkoord — nuance is hier essentieel

Volledig uitsluiten van alle event-features is een valide maar onomkeerbare beslissing. De genuanceerde academische positie is sterker:

Optie A — Volledig uitsluiten (wat jij overweegt):

"Gegeven dat 53,5% van de event-observaties (185/346) een data_confidence = 'estimated' heeft — wat betekent dat tijdstip, duur en/of schaal niet empirisch zijn geverifieerd — wordt geconcludeerd dat de event-feature-set als geheel niet voldoet aan de meetbetrouwbaarheidsnormen die een ML-model vereist voor stabiele parameterschatting. Event-features worden bijgevolg niet opgenomen in de feature engineering pipeline (nb08). Dit is een expliciete databegrenzingsbeslissing: de bevindingen van H-S4 en H-E3 blijven relevant als exploratieve observaties, maar zijn onvoldoende robuust om te vertalen naar predictieve features."

Optie B — Alleen verified events behouden (methodologisch sterker):
Behoud de ~161 geverifieerde event-dagen als binaire dummies. De cascade-features (hours_to_next_event) worden dan alleen berekend op basis van verified events. Dit is wetenschappelijk eerlijker én haalt meer signaal eruit.

Mijn aanbeveling: Als je H-E3 (cascade) al bevestigd hebt, gooi je potentieel krachtige features weg bij Optie A. Optie B is methodologisch verdedigbaarder. Maar als je de administratieve last van event-verificatie te groot vindt voor de resterende thesis-tijd, dan is Optie A verdedigbaar mits je het expliciet rapporteert als bovenstaand.


#### 3. Waar en wanneer rapporteer je de 2020-gevoeligheidsanalyse en de lag-leakage beslissing?

2020-gevoeligheidsanalyse

Wanneer: In nb09 (baseline modellen), onmiddellijk na de eerste modeltraining.

Hoe: Train hetzelfde baseline model (bijv. XGBoost of Random Forest) twee keer:

Run A: trainingsdata = 2020 + 2023 + 2024

Run B: trainingsdata = 2023 + 2024 only

Valideer beide runs op 2024 (temporele cross-validatie) via MAPE of RMSE. Als Run B significant beter valideert, is 2020-data contraproductief en gebruik je alleen 2023+2024 als definitieve trainingsset.

Academische formulering voor de thesis (methodologie-sectie):

"Gezien de structureel afwijkende patronen in de COVID-periode 2020 (H-T4, r_struct = −0.956 voor centrumtier), werd een gevoeligheidsanalyse uitgevoerd waarbij het model werd getraind met (a) 2020 + 2023 + 2024 en (b) 2023 + 2024 uitsluitend. De validatieprestaties op het jaar 2024 bepaalden de definitieve keuze van de trainingsperiode."

#### 4.Lag-leakage: eerste week 2025 (occ_lag_168h)

Wanneer: In nb08 (feature engineering), in de cel die lag-features construeert.

Wat documenteren: Bij de constructie van occ_lag_168h voor de holdout (2025) zijn de eerste 168 uur (= week 1 van januari 2025) per definitie afhankelijk van trainingsdata (december 2024). Dit is correct en geen leakage. Maar vermeld expliciet:

Dat de lag-waarden voor de eerste week van de holdout zijn geïmporteerd vanuit de laatste week van de trainingsset

Dat er geen look-ahead leakage is (toekomstige bezetting wordt nooit gebruikt)

Dat je de eerste 168 uur van de holdout expliciet hebt gecontroleerd op NaN-waarden

Academische formulering (nb08, markdown-cel):

"Tijdreeksintegriteit: occ_lag_168h voor de holdoutperiode (2025) vereist bezettingswaarden uit de laatste week van de trainingsperiode (december 2024). Deze waarden zijn aaneengesloten beschikbaar en bevatten geen missing data na de kwaliteitsfilter. Er treedt geen temporele leakage op: geen enkel lag-feature bevat bezettingsinformatie uit de toekomst ten opzichte van het voorspelmoment. De eerste 168 observaties van de holdout worden gecontroleerd op NaN-completeness vóór modelinferentie."

#### 5. Steekproefomvang per event-type & daglengte-confound dubbelchecken

- Steekproefomvang per event-type
- Daglengte confound